In [1]:
import sys, pandas as pd
print(sys.executable)
print(pd.__version__)

/home/klt/prjet-10_opclrm/.venv310/bin/python3.10
2.3.3


In [2]:
from pathlib import Path

DATA = Path("news-portal-user-interactions-by-globocom")
assert DATA.exists(), f"introuvable : {DATA.resolve()}"

for p in sorted(DATA.iterdir()):
    if p.is_file():
        print(f"{p.name}  —  {p.stat().st_size/1e6:.1f} Mo")
    else:
        n = len(list(p.rglob('*.csv')))
        print(f"[dir] {p.name}  —  {n} CSV")

articles_embeddings.pickle  —  364.0 Mo
articles_metadata.csv  —  11.1 Mo
[dir] clicks  —  385 CSV
clicks_sample.csv  —  0.1 Mo


In [3]:
import pandas as pd

meta = pd.read_csv(DATA / "articles_metadata.csv")
print(meta.shape)
print(meta.dtypes)
meta.head()

(364047, 5)
article_id       int64
category_id      int64
created_at_ts    int64
publisher_id     int64
words_count      int64
dtype: object


,article_id,category_id,created_at_ts,publisher_id,words_count
0,0,0,1513144419000,0,168
1,1,1,1405341936000,0,189
2,2,1,1408667706000,0,250
3,3,1,1408468313000,0,230
4,4,1,1407071171000,0,162


In [4]:
print("articles :", meta.article_id.nunique())
print("catégories :", meta.category_id.nunique())
print("publishers :", meta.publisher_id.nunique())
print("\nmots par article :")
print(meta.words_count.describe())

articles : 364047
catégories : 461
publishers : 1

mots par article :
count    364047.000000
mean        190.897727
std          59.502766
min           0.000000
25%         159.000000
50%         186.000000
75%         218.000000
max        6690.000000
Name: words_count, dtype: float64


In [5]:
import pickle, numpy as np

with open(DATA / "articles_embeddings.pickle", "rb") as f:
    emb = pickle.load(f)

emb = np.asarray(emb)
print("shape :", emb.shape)
print("dtype :", emb.dtype)
print("mémoire :", emb.nbytes / 1e6, "Mo")
print("aligné avec meta :", emb.shape[0] == len(meta))

shape : (364047, 250)
dtype : float32
mémoire : 364.047 Mo
aligné avec meta : True


In [6]:
sample = pd.read_csv(DATA / "clicks_sample.csv")
print(sample.shape)
print(sample.columns.tolist())
sample.head()

(1883, 12)
['user_id', 'session_id', 'session_start', 'session_size', 'click_article_id', 'click_timestamp', 'click_environment', 'click_deviceGroup', 'click_os', 'click_country', 'click_region', 'click_referrer_type']


,user_id,session_id,session_start,session_size,click_article_id,click_timestamp,click_environment,click_deviceGroup,click_os,click_country,click_region,click_referrer_type
0,0,1506825423271737,1506825423000,2,157541,1506826828020,4,3,20,1,20,2
1,0,1506825423271737,1506825423000,2,68866,1506826858020,4,3,20,1,20,2
2,1,1506825426267738,1506825426000,2,235840,1506827017951,4,1,17,1,16,2
3,1,1506825426267738,1506825426000,2,96663,1506827047951,4,1,17,1,16,2
4,2,1506825435299739,1506825435000,2,119592,1506827090575,4,1,17,1,24,2


In [ ]:
Les embeddings font 364 Mo en float32, Bien au-delà de ce qui passe sur un plan Azure gratuit. L'ACP est donc obligatoire .

In [ ]:
concaténation des clics

In [7]:
from pathlib import Path
import pandas as pd

files = sorted((DATA / "clicks").rglob("*.csv"))
print(len(files), "fichiers")

clicks = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)
print(clicks.shape)

Path("data").mkdir(exist_ok=True)
clicks.to_parquet("data/clicks_all.parquet", index=False)

385 fichiers
(2988181, 12)


In [8]:
n_users = clicks.user_id.nunique()
n_art_clicked = clicks.click_article_id.nunique()

print("interactions :", len(clicks))
print("utilisateurs :", n_users)
print("articles cliqués :", n_art_clicked, f"({n_art_clicked/len(meta):.1%} du catalogue)")
print("densité :", len(clicks) / (n_users * n_art_clicked))
print("\nclics par utilisateur :")
print(clicks.groupby("user_id").size().describe())

interactions : 2988181
utilisateurs : 322897
articles cliqués : 46033 (12.6% du catalogue)
densité : 0.00020103589647179989

clics par utilisateur :
count    322897.000000
mean          9.254285
std          14.946358
min           2.000000
25%           2.000000
50%           4.000000
75%          10.000000
max        1232.000000
dtype: float64


385 fichiers → 2 988 181 interactions, 12 colonnes.
322 897 utilisateurs, 46 033 articles cliqués.
Moyenne 9,25 clics / médiane 4 / max 1232. (L'écart entre moyenne et médiane trahit une distribution très asymétrique : une poignée d'utilisateurs très actifs tire la moyenne vers le haut, pendant que la masse en a 2 à 4)

In [ ]:
la dimension temporelle

In [9]:
clicks["click_dt"] = pd.to_datetime(clicks.click_timestamp, unit="ms")
print("du", clicks.click_dt.min(), "au", clicks.click_dt.max())
print("durée :", clicks.click_dt.max() - clicks.click_dt.min())

meta["created_dt"] = pd.to_datetime(meta.created_at_ts, unit="ms")
print("\narticles créés du", meta.created_dt.min(), "au", meta.created_dt.max())

du 2017-10-01 03:00:00.026000 au 2017-11-13 20:04:14.886000
durée : 43 days 17:04:14.860000

articles créés du 2006-09-27 11:14:35 au 2018-03-13 12:12:30


In [10]:
par_jour = clicks.set_index("click_dt").resample("D").size()
print(par_jour)

click_dt
2017-10-01     94056
2017-10-02    303177
2017-10-03    261159
2017-10-04    215415
2017-10-05    190003
2017-10-06    207646
2017-10-07    139323
2017-10-08    108110
2017-10-09    248208
2017-10-10    282391
2017-10-11    238969
2017-10-12    121467
2017-10-13    180723
2017-10-14     95216
2017-10-15     92163
2017-10-16    189779
2017-10-17     19664
2017-10-18       272
2017-10-19       125
2017-10-20       122
2017-10-21        23
2017-10-22        32
2017-10-23        40
2017-10-24        29
2017-10-25        18
2017-10-26        12
2017-10-27         7
2017-10-28         2
2017-10-29         0
2017-10-30        12
2017-10-31         4
2017-11-01         6
2017-11-02         0
2017-11-03         2
2017-11-04         2
2017-11-05         0
2017-11-06         0
2017-11-07         2
2017-11-08         0
2017-11-09         0
2017-11-10         0
2017-11-11         0
2017-11-12         0
2017-11-13         2
Freq: D, dtype: int64


Le volume s'effondre le 18 octobre. Du 1er au 17 octobre, on es entre 90 000 et 300 000 clics par jour. Le 17 tombe déjà à 19 664, puis le 18 à 272, et ensuite c'est du bruit — des journées à 0, 2, 12 clics jusqu'au 13 novembre.

Ce n'est pas une baisse d'activité réelle, c'est une coupure de collecte. Les 27 derniers jours ne contiennent presque rien.

Conséquence concrète : la fenêtre exploitable va du 1er au 16 octobre 2017, soit 16 jours. Le 17 est déjà une journée tronquée, à écarter aussi.

Deuxième point notable : le système continue de tourner, mais quelque chose a changé radicalement dans ce qui alimente le fichier. Passer de 190 000 clics le 16 à 272 le 18, ce n'est pas une baisse de trafic, aucun site ne perd 99,9 % de son audience du jour au lendemain mais ce n'est pas non plus un arrêt, puisque des lignes continuent d'arriver jusqu'au 13 novembre. Ça illustre parfaitement le problème de cold start article, et c'est un bon argument à ressortir dans la partie architecture cible.

In [11]:
tardif = clicks[clicks.click_dt >= "2017-10-18"]
print(tardif.user_id.nunique(), "users |", len(tardif), "clics")
print(tardif.click_environment.value_counts())
print(tardif.click_country.value_counts().head())

134 users | 712 clics
click_environment
4    712
Name: count, dtype: int64
click_country
1     655
10     37
6      11
2       5
9       2
Name: count, dtype: int64


In [12]:
print(clicks[clicks.click_dt < "2017-10-18"].click_environment.value_counts(normalize=True))

click_environment
4    0.971982
2    0.026692
1    0.001326
Name: proportion, dtype: float64


In [13]:
tardif = clicks[clicks.click_dt >= "2017-10-18"]
print(tardif.groupby("user_id").size().describe())
print("\nusers tardifs déjà vus avant :", 
      len(set(tardif.user_id) & set(clicks[clicks.click_dt < "2017-10-18"].user_id)))

count    134.000000
mean       5.313433
std        4.489617
min        2.000000
25%        2.000000
50%        4.000000
75%        6.750000
max       22.000000
dtype: float64

users tardifs déjà vus avant : 129


le volume chute de plus de 99 % au 17 octobre ; les interactions résiduelles proviennent quasi exclusivement d'utilisateurs déjà observés, ce qui suggère un changement de périmètre de collecte plutôt qu'une évolution du comportement. Période retenue : 1er au 16 octobre 2017.

In [ ]:
filtrer et couper

In [14]:
import pandas as pd

FIN = pd.Timestamp("2017-10-17")
clicks_ok = clicks[clicks.click_dt < FIN].copy()
print("avant :", len(clicks), "→ après :", len(clicks_ok))

CUT = pd.Timestamp("2017-10-14")
train = clicks_ok[clicks_ok.click_dt < CUT]
test  = clicks_ok[clicks_ok.click_dt >= CUT]

print("\ntrain :", len(train), "|", train.user_id.nunique(), "users")
print("test  :", len(test), "|", test.user_id.nunique(), "users")

avant : 2988181 → après : 2967805

train : 2590647 | 302101 users
test  : 377158 | 95773 users


In [15]:
users_test = set(test.user_id)
users_train = set(train.user_id)

connus = users_test & users_train
print("users test connus du train :", len(connus), f"({len(connus)/len(users_test):.1%})")
print("users test en cold start   :", len(users_test - users_train))

users test connus du train : 76002 (79.4%)
users test en cold start   : 19771


In [ ]:
la logique 
la liste des articles lus par chaque personne.

moi : articles 5, 12, 30
Paul : articles 5, 12, 30, 47
Marie : articles 100, 200, 300

Paul me ressemble, on a trois articles en commun. Marie non, aucun. Donc on me propose l'article 47, que Paul a lu et pas moi.

In [16]:
r = train.groupby(["user_id", "click_article_id"]).size()
print(r.value_counts().head())

1    2528127
2      27678
3       1624
4        278
5         85
Name: count, dtype: int64


In [17]:
import numpy as np

nb_clics = train.groupby("user_id").size()
eligibles = nb_clics[nb_clics >= 5].index
print("utilisateurs éligibles :", len(eligibles))

rng = np.random.default_rng(42)
echantillon = rng.choice(eligibles, size=min(5000, len(eligibles)), replace=False)
print("échantillon :", len(echantillon))

utilisateurs éligibles : 145586
échantillon : 5000


In [18]:
hist = (train[train.user_id.isin(echantillon)]
        .groupby("user_id")["click_article_id"]
        .apply(list)
        .to_dict())

u = echantillon[0]
print("user", u, "→", len(hist[u]), "articles :", hist[u][:10])

user 99653 → 6 articles : [160417, 158536, 235230, 83893, 235440, 338350]


In [19]:
norms = np.linalg.norm(emb, axis=1, keepdims=True)
emb_norm = emb / np.where(norms == 0, 1, norms)
print(emb_norm.shape)

(364047, 250)


In [20]:
def recommander(user_id, k=5):
    articles_lus = hist[user_id]
    profil = emb_norm[articles_lus].mean(axis=0)
    profil = profil / np.linalg.norm(profil)

    scores = emb_norm @ profil
    scores[articles_lus] = -np.inf

    top = np.argpartition(-scores, k)[:k]
    return top[np.argsort(-scores[top])]

In [21]:
u = echantillon[0]
reco = recommander(u)

print("A LU :")
print(meta.loc[hist[u], ["article_id", "category_id", "words_count"]])
print("\nON PROPOSE :")
print(meta.loc[reco, ["article_id", "category_id", "words_count"]])

A LU :
        article_id  category_id  words_count
160417      160417          281          173
158536      158536          281          858
235230      235230          375          262
83893        83893          174          180
235440      235440          375          211
338350      338350          437          177

ON PROPOSE :
        article_id  category_id  words_count
103066      103066          228          274
235294      235294          375          178
158543      158543          281          211
107335      107335          228          146
98292        98292          221          144


In [22]:
import numpy as np

taux = []
for u in echantillon[:200]:
    cats_lues = set(meta.loc[hist[u], "category_id"])
    cats_reco = meta.loc[recommander(u), "category_id"]
    taux.append(cats_reco.isin(cats_lues).mean())

print(f"recouvrement moyen des catégories : {np.mean(taux):.1%}")

recouvrement moyen des catégories : 72.9%


In [23]:
def recommander_dernier(user_id, k=5):
    lus = hist[user_id]
    dernier = lus[-1]
    scores = emb_norm @ emb_norm[dernier]
    scores[lus] = -np.inf
    top = np.argpartition(-scores, k)[:k]
    return top[np.argsort(-scores[top])]

print("moyenne :", recommander(u))
print("dernier :", recommander_dernier(u))


moyenne : [353599 279733 285777  96199 281359]
dernier : [118443 118444 148917 148058 288457]


In [24]:
np.save("data/emb_norm.npy", emb_norm.astype(np.float32))
print("sauvegardé :", emb_norm.nbytes / 1e6, "Mo")

sauvegardé : 364.047 Mo


In [25]:
from scipy.sparse import csr_matrix
import numpy as np

sub = train[train.user_id.isin(echantillon)]

users = np.sort(sub.user_id.unique())
items = np.sort(sub.click_article_id.unique())

u_idx = {u: i for i, u in enumerate(users)}
i_idx = {a: i for i, a in enumerate(items)}

rows = sub.user_id.map(u_idx).values
cols = sub.click_article_id.map(i_idx).values
vals = np.ones(len(sub), dtype=np.float32)

mat = csr_matrix((vals, (rows, cols)), shape=(len(users), len(items)))
mat.sum_duplicates()

print(mat.shape, "|", mat.nnz, "interactions")
print("densité :", mat.nnz / (mat.shape[0] * mat.shape[1]))

(5000, 6673) | 73992 interactions
densité : 0.0022176532294320398


In [26]:
from implicit.als import AlternatingLeastSquares

model = AlternatingLeastSquares(
    factors=50,
    regularization=0.05,
    iterations=20,
    random_state=42,
)
model.fit(mat)
print("terminé")

/home/klt/prjet-10_opclrm/.venv310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/klt/prjet-10_opclrm/.venv310/lib/python3.10/site-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 12 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
100%|██████████████████| 20/20 [00:00<00:00, 37.67it/s]

terminé


In [27]:
def recommander_collab(user_id, k=5):
    ui = u_idx[user_id]
    ids, scores = model.recommend(ui, mat[ui], N=k, filter_already_liked_items=True)
    return [items[i] for i in ids], scores

In [28]:
uid = echantillon[0]

print("A LU :")
print(meta.loc[hist[uid], ["article_id", "category_id"]].to_string(index=False))

print("\nCONTENT-BASED :")
print(meta.loc[recommander(uid), ["article_id", "category_id"]].to_string(index=False))

arts, sc = recommander_collab(uid)
print("\nCOLLABORATIF :")
print(meta.loc[arts, ["article_id", "category_id"]].to_string(index=False))
print("scores :", sc.round(3))

A LU :
 article_id  category_id
     160417          281
     158536          281
     235230          375
      83893          174
     235440          375
     338350          437

CONTENT-BASED :
 article_id  category_id
     103066          228
     235294          375
     158543          281
     107335          228
      98292          221

COLLABORATIF :
 article_id  category_id
     313431          431
     156964          281
      70646          136
     156619          281
     119193          247
scores : [0.179 0.158 0.148 0.13  0.127]


L'utilisateur a lu en catégories 281, 375, 174, 437.

Le content-based propose 228, 375, 281, 228, 221 — dont deux catégories exactement identiques à ses lectures.

Le collaboratif propose 431, 281, 136, 281, 247 — une seule catégorie commune (281, deux fois), et il part vers 431, 136, 247 que l'utilisateur n'a jamais touchées.

C'est exactement le comportement attendu. Le collaboratif ne voit pas le contenu : il a repéré que des lecteurs au parcours semblable ont lu ces articles-là, et il les propose sans savoir de quoi ils parlent. D'où plus de découverte, mais aussi plus de risque de hors-sujet.

In [29]:
import pickle, numpy as np

np.save("data/als_user_factors.npy", model.user_factors.astype(np.float32))
np.save("data/als_item_factors.npy", model.item_factors.astype(np.float32))

with open("data/als_mappings.pkl", "wb") as f:
    pickle.dump({"users": users, "items": items}, f)

print("user_factors :", model.user_factors.shape)
print("item_factors :", model.item_factors.shape)

user_factors : (5000, 50)
item_factors : (6673, 50)
